#### Install required libraries

In [1]:
!pip install -qU langchain-google-genai langchain-community langchain-huggingface langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/113.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip install -qU langchain-google-genai

In [3]:
!pip install -qU langchain-text-splitters

In [4]:
!pip install -q pypdf langchain langchain-community sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 35.4 MB/s eta 0:00:00


## Part 1 — Data Understanding & Preprocessing
We will load the PDF, extract text, and split it into chunks.

##### 1.Load Dataset

In [5]:
import pypdf
# Updated import for modern langchain versions
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = '/content/intro-to-ml.pdf'
reader = pypdf.PdfReader(pdf_path)
print(reader)

##### 2. Extract text from all pages

In [6]:
text_content = ""
for page in reader.pages:
    text_content += page.extract_text() + "\n"

##### 3.Explore document structure

In [7]:
import re

# a. Number of pages
num_pages = len(reader.pages)
print(f"Total Pages: {num_pages}")

# b. Text quality issues etc.
print("\n--- Text Quality Analysis ---")

# Check for potential empty extraction
if not text_content.strip():
    print("Warning: Extracted text is empty. Check if the PDF is image-based (OCR required).")
else:
    print(f"Total characters extracted: {len(text_content)}")

# Check for unusual character patterns (e.g., lack of whitespace or low alphanumeric ratio)
alnum_count = sum(c.isalnum() for c in text_content)
alnum_ratio = alnum_count / len(text_content) if len(text_content) > 0 else 0
print(f"Alphanumeric character ratio: {alnum_ratio:.2f}")

# Check for empty pages
empty_pages = []
for i, page in enumerate(reader.pages):
    if not page.extract_text().strip():
        empty_pages.append(i + 1)

if empty_pages:
    print(f"Found {len(empty_pages)} potentially empty pages: {empty_pages[:10]}...")
else:
    print("No empty pages detected.")

print(f"\nFirst 500 characters of text for manual inspection:\n{'-'*40}\n{text_content[:500]}\n{'-'*40}")

Total Pages: 392

--- Text Quality Analysis ---
Total characters extracted: 692487
Alphanumeric character ratio: 0.75
Found 5 potentially empty pages: [2, 144, 224, 318, 336]...

First 500 characters of text for manual inspection:
----------------------------------------
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


Andreas C. Müller and Sarah Guido
Introduction to Machine Learning
with Python
A Guide for Data Scientists
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing
978-1-449-36941-5
[LSI]
Introduction to Machine Learning with Python
by Andreas C. Müller and Sarah Guido
Copyright © 2017 Sarah Guido, Andreas Müller. All rights reserved.
Printed in the United S
----------------------------------------


##### 4. Split text into chunks

In [8]:
# Using suggested chunk size of 1000 with 100 overlap
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)
chunks = text_splitter.split_text(text_content)

print(f"\nTotal Chunks created: {len(chunks)}")


Total Chunks created: 782


## Part 2 — Embedding & Vector Database


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

batch_size = 100
db = None

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]

    if db is None:
        db = FAISS.from_texts(batch, embeddings)
    else:
        db.add_texts(batch)

vector_db = db

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Description of embedding model used:**
We are using the `all-MiniLM-L6-v2` SentenceTransformer model via LangChain. This model is a 'MiniLM' (Mini Language Model) that maps sentences and paragraphs to a 384-dimensional dense vector space. It is specifically designed to balance speed and performance for semantic search tasks.

**Explanation of vector storage approach:**
We are using `FAISS` (Facebook AI Similarity Search) as our vector database. FAISS is a library for efficient similarity search and clustering of dense vectors. It works by creating an index of the chunk embeddings, allowing us to perform fast 'Nearest Neighbor' searches to find the most relevant context for a given user query.

##### 3. Implement similarity search mechanism (Example of retrieved chunks)

In [10]:
sample_query = "What is the main goal of supervised learning?"
k = 3

# Perform similarity search
docs = vector_db.similarity_search(sample_query, k=k)

print(f"Sample Query: {sample_query}\n")
for i, doc in enumerate(docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(f"{doc.page_content[:300]}...\n")

Sample Query: What is the main goal of supervised learning?

--- Retrieved Chunk 1 ---
example you only observe traffic, and you don’t know what constitutes normal
and abnormal behavior, this is an unsupervised problem.
For both supervised and unsupervised learning tasks, it is important to have a repre‐
sentation of your input data that a computer can understand. Often it is helpful ...

--- Retrieved Chunk 2 ---
of an early version of this book, and helped me shape it in many ways.
On the personal side, I want to thank my parents, Harald and Margot, and my sister,
Miriam, for their continuing support and encouragement. I also want to thank the
many people in my life whose love and friendship gave me the ene...

--- Retrieved Chunk 3 ---
it has never seen before without any help from a human. Going back to our example
of spam classification, using machine learning, the user provides the algorithm with a
large number of emails (which are the input), together with information about
whet

## Part 3 — Retrieval Pipeline
Now we test the retrieval by converting a query into an embedding and finding the top-k most similar chunks.

In [11]:
query = "What is supervised learning?"
k = 5  # Number of relevant chunks to retrieve

# Perform similarity search
retrieved_docs = vector_db.similarity_search(query, k=k)

print(f"Query: {query}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:500] + "...\n")

Query: What is supervised learning?

--- Chunk 1 ---
it has never seen before without any help from a human. Going back to our example
of spam classification, using machine learning, the user provides the algorithm with a
large number of emails (which are the input), together with information about
whether any of these emails are spam (which is the desired output). Given a new
email, the algorithm will then produce a prediction as to whether the new email is
spam.
Machine learning algorithms that learn from input/output pairs are called supervised...

--- Chunk 2 ---
example you only observe traffic, and you don’t know what constitutes normal
and abnormal behavior, this is an unsupervised problem.
For both supervised and unsupervised learning tasks, it is important to have a repre‐
sentation of your input data that a computer can understand. Often it is helpful to
think of your data as a table. Each data point that you want to reason about (each
email, each customer, each transaction) 

In [12]:
query = "What is supervised learning?"
k = 3  # changed to 3

# Perform similarity search
retrieved_docs = vector_db.similarity_search(query, k=k)

print(f"Query: {query}\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:500] + "...\n")

Query: What is supervised learning?

--- Chunk 1 ---
it has never seen before without any help from a human. Going back to our example
of spam classification, using machine learning, the user provides the algorithm with a
large number of emails (which are the input), together with information about
whether any of these emails are spam (which is the desired output). Given a new
email, the algorithm will then produce a prediction as to whether the new email is
spam.
Machine learning algorithms that learn from input/output pairs are called supervised...

--- Chunk 2 ---
example you only observe traffic, and you don’t know what constitutes normal
and abnormal behavior, this is an unsupervised problem.
For both supervised and unsupervised learning tasks, it is important to have a repre‐
sentation of your input data that a computer can understand. Often it is helpful to
think of your data as a table. Each data point that you want to reason about (each
email, each customer, each transaction) 

## Part 4 — Answer Generation (RAG)
We will use the Gemini API (via LangChain) to generate answers grounded in the retrieved context.


**Prompt Strategy:**
We have designed a 'System Prompt' that acts as a guardrail for the Gemini 1.5 Flash model.
1. **Context Injection:** The `{context}` variable is populated only with the top-5 most relevant chunks from our FAISS vector store.
2. **Constraint Clause:** The instruction *"If you don't know the answer, say you don't know"* is a critical technique in RAG to minimize hallucinations. It prevents the model from using its internal training data to 'guess' when the provided PDF context is insufficient.
3. **Formatting:** We use a structured `ChatPromptTemplate` to maintain a clear distinction between system instructions and user queries, which helps the model follow grounding rules more effectively.

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata

# Initialize the Gemini LLM
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

In [22]:
# Define the prompt template for RAG
template = """You are an assistant for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
If you don't know the answer, say that you don't know. \
Use three sentences maximum and keep the answer concise.
Question: {input}
Context: {context}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [23]:
# Create the RAG chain
rag_chain = (
    {"context": vector_db.as_retriever(), "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Now, let's test the `rag_chain` with a sample query to ensure it generates answers based on the retrieved context.

In [24]:
sample_question = "What is supervised learning?"
response = rag_chain.invoke(sample_question)

print(f"Question: {sample_question}")
print(f"Answer: {response}")

Question: What is supervised learning?
Answer: Supervised learning algorithms are a type of machine learning that learn from input/output pairs. A "teacher" provides supervision by giving the desired outputs for each example the algorithm learns from. For instance, in spam classification, the algorithm is given emails (input) and information about whether they are spam (desired output).


## Part 5 — End-to-End RAG Application
This final section integrates everything into a function for real-time Q&A.

In [26]:
def ask_question(question):
    print(f"Question: {question}")
    # The rag_chain expects the question string directly as input
    response = rag_chain.invoke(question)
    # StrOutputParser returns a string directly, so 'response' is the answer
    print(f"Answer: {response}\n")

# Demonstration with multiple queries
queries = [
    "What is the difference between supervised and unsupervised learning?",
    "How do neural networks work?",
    "Explain the concept of over-fitting."
]

if 'rag_chain' in locals():
    for q in queries:
        ask_question(q)
else:
    print("Please run the initialization cell (Part 4) first.")

Question: What is the difference between supervised and unsupervised learning?
Answer: Unsupervised learning algorithms operate with no known output or "teacher" to guide the learning process, meaning only the input data is provided. The algorithm's goal is to extract knowledge directly from this input data. In contrast, supervised learning involves a known output or target variable that the algorithm learns to predict based on the given input data.

Question: How do neural networks work?
Answer: Neural networks, specifically multilayer perceptrons (MLPs), work by performing multiple stages of processing, generalizing from linear models. They operate by repeatedly computing weighted sums of inputs. This process first calculates "hidden units" representing an intermediate step, which are then combined via further weighted sums to produce the final output.

Question: Explain the concept of over-fitting.
Answer: Overfitting occurs when a model is too complex for the amount of information 